In [1]:
# %pip install langchain_openai langchain_community langchain pymysql chromadb -q

In [1]:
db_user = "root"
db_host = "localhost"
db_password = "lavi9755"
db_name = "classicmodels"

In [2]:
from langchain_community.utilities.sql_database import SQLDatabase
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}")


In [3]:
%pip install google_genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# %pip install google_generativeai

In [5]:

import google.generativeai as genai
import pymysql

# Gemini setup
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

# DB connection
conn = pymysql.connect(
    host="localhost",
    user="root",
    password= db_password,
    database= db_name
)
cursor = conn.cursor()

c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23328\1246155490.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [8]:
print(db.dialect)
print(db.get_usable_table_names)
print(db.table_info)

mysql
<bound method SQLDatabase.get_usable_table_names of <langchain_community.utilities.sql_database.SQLDatabase object at 0x000002B6EEFAE780>>

CREATE TABLE sales_raw (
	order_id VARCHAR(10), 
	customer_name VARCHAR(100), 
	product_category VARCHAR(50), 
	order_date TEXT, 
	quantity TEXT, 
	price_per_unit TEXT, 
	country VARCHAR(50)
)DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB

/*
3 rows from sales_raw table:
order_id	customer_name	product_category	order_date	quantity	price_per_unit	country
O001	   John Doe   	Electronics	2025-08-10	2	500	India
O002	Jane Smith	HomeAppliance	08/11/2025	3	1200	 United States 
O003	John Doe	Electronics	2025-08-12	two	500	India
*/


In [9]:
# %pip install langchain-google-genai

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0
)

ImportError: cannot import name 'LangSmithParams' from 'langchain_core.language_models' (c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_core\language_models\__init__.py)

In [22]:
def get_schema():
    schema = ""
    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()

    for (table_name,) in tables:
        cursor.execute(f"DESCRIBE {table_name}")
        columns = cursor.fetchall()

        schema += f"\nTable: {table_name}\n"
        for col in columns:
            schema += f"{col[0]} ({col[1]})\n"

    return schema

In [23]:
print(get_schema())


Table: sales_raw
order_id (varchar(10))
customer_name (varchar(100))
product_category (varchar(50))
order_date (text)
quantity (text)
price_per_unit (text)
country (varchar(50))



In [24]:
def generate_query(question):
    schema = get_schema()

    prompt = f"""
You are an expert MySQL developer.

Database schema:
{schema}

Rules:
- Only generate SELECT queries
- Use correct table and column names
- Do not explain anything

Question: {question}

SQL Query:
"""

    response = model.generate_content(prompt)

    query = response.text.strip()
    query = query.replace("```sql", "").replace("```", "").strip()

    return query

In [25]:
def run_query(query):
    if any(word in query.lower() for word in ["drop", "delete", "update", "insert"]):
        return "🚫 Unsafe query blocked"

    try:
        cursor.execute(query)
        return cursor.fetchall()
    except Exception as e:
        return f"Error: {str(e)}"

In [27]:
def ask(question):
    query = generate_query(question)
    print("Generated SQL:", query)

    result = run_query(query)

    return result

In [20]:
# import google.generativeai as genai
# from dotenv import load_dotenv
# import os

# load_dotenv()
# GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
# genai.configure(api_key=GOOGLE_API_KEY)

# for m in genai.list_models():
#     print(m.name, m.supported_generation_methods)

models/gemini-2.5-flash ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite-001 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-tts ['countTokens', 'generateContent']
models/gemini-2.5-pro-preview-tts ['countTokens', 'generateContent', 'batchGenerateContent']
models/gemma-3-1b-it ['generateContent', 'countTokens']
models/gemma-3-4b-it ['generateContent', 'countTokens']
models/gemma-3-12b-it ['generateContent', 'countTokens']
models/gemma-3-

In [28]:
print(ask("How many records are there?"))

NotFound: 404 models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.

In [35]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import initialize_agent, AgentType

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

agent = initialize_agent(
    tools=toolkit.get_tools(),
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

agent.run("How many records are there?")

ImportError: cannot import name 'initialize_agent' from 'langchain.agents' (c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain\agents\__init__.py)

In [ ]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
execute_query = QuerySQLDataBaseTool(db=db)
execute_query.invoke(query)


In [ ]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

answer_prompt = PromptTemplate.from_template(
     """Given the following user question, corresponding SQL query, and SQL result, answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
 Answer: """
 )

rephrase_answer = answer_prompt | llm | StrOutputParser()

chain = (
     RunnablePassthrough.assign(query=generate_query).assign(
         result=itemgetter("query") | execute_query
     )
     | rephrase_answer
 )

chain.invoke({"question": "How many customers have an order count greater than 5"})


In [7]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
execute_query = QuerySQLDataBaseTool(db=db)
execute_query.invoke(query)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_12444\1101547050.py:2: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  execute_query = QuerySQLDataBaseTool(db=db)


NameError: name 'query' is not defined

### creating prompt and chain

In [ ]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

 answer_prompt = PromptTemplate.from_template(
     """Given the following user question, corresponding SQL query, and SQL result, answer the user question.

 Question: {question}
 SQL Query: {query}
 SQL Result: {result}
 Answer: """
 )

 rephrase_answer = answer_prompt | llm | StrOutputParser()

 chain = (
     RunnablePassthrough.assign(query=generate_query).assign(
         result=itemgetter("query") | execute_query
     )
     | rephrase_answer
 )

 chain.invoke({"question": "How many customers have an order count greater than 5"})


In [8]:
chain.get_prompts()[0].pretty_print()

NameError: name 'chain' is not defined